# Exercise 05 — Building a Transformer

In the previous notebook you built attention, masks and positional encodings as separate pieces. Here you put them together into the full **encoder–decoder Transformer** of [Attention Is All You Need](https://arxiv.org/abs/1706.03762), train it as a small translation model, and look at what its attention heads have learned.

Training a Transformer to translate between two languages takes a GPU and a lot of patience, so we translate something smaller: **dates**. The model reads a date the way a person would write it and has to produce the ISO format:

```
saturday, 12 september 2026   ->   2026-09-12
```

It is a real sequence-to-sequence problem — input and output have different lengths, the order of the parts changes, and words have to be turned into numbers — but a tiny model solves it, and **training takes one to two minutes on a CPU**. There is nothing to download. In notebook 2 you work with a full-size Transformer that translates English to French.

## What you will do
1. **The task** — generate the data and turn dates into tensors.
2. **Multi-head attention** — several attention heads in parallel, and a check against PyTorch's own implementation.
3. **Encoder and decoder layers** — self-attention, cross-attention, feed-forward, residual connections and layer norm.
4. **The full model** — embeddings, positions, the masks, and two tests that show whether your masks work.
5. **Training and decoding** — teacher forcing, overfitting a single batch, the training run, and greedy decoding.
6. **What did the model learn?** — visualise the cross-attention, and try to break the model.
7. **Optional: the same model with `nn.Transformer`.**
8. **Multiple-choice questions** — to check your understanding.

## How to work through it
- Run the cells **in order** and fill in **every `# TODO`**.
- Tasks are numbered (**1.1**, **1.2**, …). Tasks that say *Your answer here* want a short written answer, not code.
- Most implementation tasks are followed by a **✅ Check** cell that verifies your work automatically. Run it and make sure it passes before you move on.

> **Tip — debugging a Transformer.** Almost every bug in a Transformer is a **shape** or a **mask** bug, and the model will often still train, just badly. Print shapes while you work, and take the check cells seriously: the ones in Part 4 and Part 5 are exactly the tests you would write for your own model.

In [ ]:
%matplotlib inline
import datetime
import math
import random

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)

# Use the GPU if there is one; a CPU is enough for this notebook.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

## 1. The task: translating dates

The cell below generates the data. Every example is a random date between 1900 and 2099, written in one of seven formats (the **source**), together with its ISO form `YYYY-MM-DD` (the **target**).

In [ ]:
MONTHS = ["january", "february", "march", "april", "may", "june",
          "july", "august", "september", "october", "november", "december"]
WEEKDAYS = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]

FORMATS = [
    lambda d: f"{d.day} {MONTHS[d.month - 1]} {d.year}",                            # 12 september 2026
    lambda d: f"{MONTHS[d.month - 1]} {d.day}, {d.year}",                           # september 12, 2026
    lambda d: f"{WEEKDAYS[d.weekday()]}, {d.day} {MONTHS[d.month - 1]} {d.year}",   # saturday, 12 september 2026
    lambda d: f"{d.day} {MONTHS[d.month - 1][:3]} {d.year}",                        # 12 sep 2026
    lambda d: f"{MONTHS[d.month - 1][:3]} {d.day} {d.year}",                        # sep 12 2026
    lambda d: f"{d.day:02d}/{d.month:02d}/{d.year}",                                # 12/09/2026
    lambda d: f"{d.day}.{d.month}.{d.year}",                                        # 12.9.2026
]

FIRST_DAY = datetime.date(1900, 1, 1).toordinal()
LAST_DAY = datetime.date(2099, 12, 31).toordinal()


def make_pair():
    # A random date as (source, target), e.g. ("12 sep 2026", "2026-09-12").
    date = datetime.date.fromordinal(random.randint(FIRST_DAY, LAST_DAY))
    return random.choice(FORMATS)(date), date.isoformat()


train_pairs = [make_pair() for _ in range(20_000)]
test_pairs = [make_pair() for _ in range(1_000)]

for source, target in train_pairs[:8]:
    print(f"{source:<32} -> {target}")

### From text to tensors

The model works on **characters**. The vocabulary is small: the letters that occur in month and weekday names, the digits, a few punctuation marks, and three special tokens:

- `<pad>` (id 0) fills up shorter sequences in a batch,
- `<s>` (id 1) marks the beginning of a sequence — it is also what the decoder is given to start generating,
- `</s>` (id 2) marks the end — it is how the decoder says "I am done".

**1.1 Complete `encode` and `make_batch`.**

Hints:
- `encode` wraps the character ids in `BOS` and `EOS`.
- In `make_batch`, the sources have different lengths: fill the shorter ones with `PAD` up to the length of the longest one. The targets all have the same length (why?), so they need no padding.

In [ ]:
PAD, BOS, EOS = 0, 1, 2
SPECIAL_TOKENS = ["<pad>", "<s>", "</s>"]

characters = sorted(set("".join(MONTHS + WEEKDAYS) + "0123456789 ,/.-"))
itos = SPECIAL_TOKENS + characters             # id -> token
stoi = {token: i for i, token in enumerate(itos)}  # token -> id
VOCAB_SIZE = len(itos)
print(f"{VOCAB_SIZE} tokens:", itos)


def encode(text):
    # "12 sep" -> [BOS, id of "1", id of "2", ..., EOS]
    return [BOS] + [stoi[c] for c in text] + [EOS]


def decode(ids):
    # The inverse of encode; special tokens are dropped.
    return "".join(itos[i] for i in ids if i >= len(SPECIAL_TOKENS))


def make_batch(pairs):
    # Turn a list of (source, target) pairs into two tensors of token ids:
    # src (batch_size, src_len), padded with PAD, and tgt (batch_size, tgt_len).
    src = [encode(source) for source, _ in pairs]
    tgt = [encode(target) for _, target in pairs]
    src_len = max(len(ids) for ids in src)
    src = [ids + [PAD] * (src_len - len(ids)) for ids in src]
    return torch.tensor(src, device=device), torch.tensor(tgt, device=device)


src, tgt = make_batch(train_pairs[:3])
print(src)
print(tgt)

In [ ]:
# ✅ Check your encoding
ids = encode("1 may")
assert ids[0] == BOS and ids[-1] == EOS and len(ids) == 7, "encode should return BOS + one id per character + EOS"
assert decode(ids) == "1 may", "decode(encode(text)) should give back the text"

src, tgt = make_batch([("1 may 2000", "2000-05-01"), ("wednesday, 27 september 2034", "2034-09-27")])
assert src.shape == (2, 30) and tgt.shape == (2, 12), "expected src (2, 30) and tgt (2, 12)"
assert src[0, 11] == EOS and torch.all(src[0, 12:] == PAD), "the shorter source should be filled up with PAD after its EOS"
assert decode(src[0].tolist()) == "1 may 2000" and decode(tgt[1].tolist()) == "2034-09-27"
print("Looks good ✅")

**1.2 What makes this task harder than it looks?** Compare a few sources with their targets and name at least three things the model has to learn. Why would the models from last week (one label per sequence) or a model that labels every input character separately not work here?

---

*Answer:*

- **Reordering.** The year comes last in every source and first in the target. To write the first character of the output the model has to look at the *end* of the input, and where exactly that is depends on the format and on the length of the month name.
- **Words to numbers.** `september` and `sep` have to become `09`. That is a lookup table the model has to memorise, and it can only tell `march` from `may` and `june` from `july` by their third or fourth letter.
- **Variable field widths.** `3.4.2021` and `13.10.2021` both have to come out with two-digit fields, so the model has to insert leading zeros — and notice whether a second digit follows before it writes the first one.
- **Day and month look alike.** `04/03/2021` is the 4th of March (day first), while in `mar 4 2021` the day comes second. Which number is the day depends on the format.
- **Ignoring things.** The weekday carries no information that is needed for the output.

Input and output have **different lengths** and **no one-to-one alignment** between positions, so neither a single label per sequence nor one label per input character fits. We need a model that reads the whole input first and then writes an output of whatever length it takes, one token at a time, looking back at the relevant part of the input for every token it writes: an **encoder–decoder with attention**.

---

## 2. Multi-head attention

One attention head computes one set of weights: for every query, *one* distribution over the keys. But a word may want to look at several things at once, for different reasons. **Multi-head attention** runs several heads in parallel, each with its own learned projections, and concatenates their outputs.

In practice the heads are not separate modules. The input is projected once to `embed_dim` dimensions, and the result is **cut into `num_heads` chunks** of size `head_dim = embed_dim / num_heads`:

```
(batch, seq_len, embed_dim)
    -> view      (batch, seq_len, num_heads, head_dim)
    -> transpose (batch, num_heads, seq_len, head_dim)      # the heads now act like an extra batch dimension
```

Your attention function from the previous notebook works on the last two dimensions, so it handles all heads of all examples at once. Afterwards the two steps are undone to glue the heads back together.

**2.1 Complete the `forward` method of `MultiHeadAttention`.**

Hints:
- Use `-1` for the sequence length in `view`. Queries and keys can have different lengths (in cross-attention they do), and `-1` takes care of that.
- The mask is boolean, `True` where attention is **allowed** — exactly like `masked_attention` in the previous notebook.
- After `transpose` a tensor is no longer contiguous in memory, and `view` refuses to work on it. Use `.reshape(...)` instead, or `.contiguous().view(...)`.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # Projections of the input to queries, keys and values (for all heads at once)
        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.value_proj = nn.Linear(embed_dim, embed_dim)
        # Final projection after the heads have been concatenated
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, query, key, value, mask=None):
        # query: (batch_size, q_len, embed_dim)    key, value: (batch_size, k_len, embed_dim)
        # mask: boolean, broadcastable to (batch_size, num_heads, q_len, k_len); True = may attend
        # Returns the output (batch_size, q_len, embed_dim) and the weights (batch_size, num_heads, q_len, k_len).
        batch_size = query.size(0)

        # 1) Linear projections -> (batch_size, seq_len, embed_dim)
        Q = self.query_proj(query)
        K = self.key_proj(key)
        V = self.value_proj(value)

        # 2) Split into heads -> (batch_size, num_heads, seq_len, head_dim)
        Q = Q.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # 3) Scaled dot-product attention, for all heads at once
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.head_dim)  # (batch_size, num_heads, q_len, k_len)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        attn_weights = F.softmax(scores, dim=-1)
        attn_output = attn_weights @ V                               # (batch_size, num_heads, q_len, head_dim)

        # 4) Concatenate the heads -> (batch_size, q_len, embed_dim)
        attn_output = attn_output.transpose(1, 2).reshape(batch_size, -1, self.embed_dim)

        # 5) Final projection
        output = self.out_proj(attn_output)

        return output, attn_weights

The check below copies the weights of your module into PyTorch's own `nn.MultiheadAttention` and compares the two. If every step of your `forward` is right, outputs and attention weights agree to the last digit.

In [ ]:
# ✅ Check your multi-head attention against nn.MultiheadAttention
torch.manual_seed(0)
mha = MultiHeadAttention(embed_dim=32, num_heads=4)
reference = nn.MultiheadAttention(embed_dim=32, num_heads=4, batch_first=True)
with torch.no_grad():
    reference.in_proj_weight.copy_(torch.cat([mha.query_proj.weight, mha.key_proj.weight, mha.value_proj.weight]))
    reference.in_proj_bias.copy_(torch.cat([mha.query_proj.bias, mha.key_proj.bias, mha.value_proj.bias]))
    reference.out_proj.weight.copy_(mha.out_proj.weight)
    reference.out_proj.bias.copy_(mha.out_proj.bias)

q = torch.randn(2, 5, 32)   # 5 queries ...
kv = torch.randn(2, 7, 32)  # ... attending to 7 keys, as in cross-attention
output, attn_weights = mha(q, kv, kv)
ref_output, ref_weights = reference(q, kv, kv, average_attn_weights=False)

assert output.shape == (2, 5, 32), f"the output should have shape (2, 5, 32), got {tuple(output.shape)}"
assert attn_weights.shape == (2, 4, 5, 7), f"the weights should have shape (2, 4, 5, 7), got {tuple(attn_weights.shape)}"
assert torch.allclose(attn_weights, ref_weights, atol=1e-5), "the attention weights differ from PyTorch's — check the split into heads and the scaling (sqrt of head_dim!)"
assert torch.allclose(output, ref_output, atol=1e-5), "the weights are right but the output differs — check how you concatenate the heads"

mask = torch.ones(2, 1, 1, 7, dtype=torch.bool)
mask[:, :, :, -2:] = False  # hide the last two keys
_, masked_weights = mha(q, kv, kv, mask=mask)
assert torch.all(masked_weights[..., -2:] == 0), "masked keys must get a weight of exactly 0"
assert torch.allclose(masked_weights.sum(dim=-1), torch.ones(2, 4, 5)), "the weights must still sum to 1"
print("Looks good ✅")

**2.2 How many parameters does `MultiHeadAttention(embed_dim=64, num_heads=4)` have?** How does that number change with 8 heads, or with a single head? What *does* change when you use more heads?

You can check your answer with `sum(p.numel() for p in module.parameters())`.

---

*Answer:*

Four linear layers of $64 \times 64$ weights plus 64 biases each: $4 \cdot (64 \cdot 64 + 64) = $ **16,640**.

The number of heads does **not** appear in that calculation: 1, 4 or 8 heads all have 16,640 parameters. The heads are not extra capacity, they are a different way of *using* the same projections. The 64 dimensions are cut into `num_heads` chunks, so more heads means **more attention patterns, each working in a smaller space** (`head_dim` = 64, 16 or 8). With one head, a position gets one distribution over the keys; with 8 heads it can look at 8 different places for 8 different reasons, but each head compares queries and keys in only 8 dimensions. The compute is the same as well.

---

## 3. Encoder and decoder layers

A Transformer is a stack of identical layers, and every layer is built from the same kind of **sub-layer**:

```
x = LayerNorm(x + SubLayer(x))
```

The `x + ...` is a **residual connection**: the sub-layer only computes an *update* to `x`. The layer norm keeps the scale of the activations under control.

An **encoder layer** has two sub-layers: self-attention, and a position-wise feed-forward network (`Linear → ReLU → Linear`, applied to every position separately). It is given below — read it carefully, because the decoder layer follows the same pattern.

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_dim):
        super().__init__()
        self.self_attn = MultiHeadAttention(embed_dim, num_heads)
        self.linear1 = nn.Linear(embed_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, embed_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x, mask=None):
        # x: (batch_size, src_len, embed_dim)

        # 1) Self-attention: queries, keys and values all come from x
        attn_output, _ = self.self_attn(x, x, x, mask=mask)
        x = self.norm1(x + attn_output)

        # 2) Feed-forward
        ff_output = self.linear2(F.relu(self.linear1(x)))
        x = self.norm2(x + ff_output)

        return x

A **decoder layer** has three sub-layers:

1. **Masked self-attention** over the target tokens written so far (with the causal mask).
2. **Cross-attention**: the decoder asks the questions, the encoder provides the answers. This is the only place where information flows from the source to the target.
3. The **feed-forward** network.

**3.1 Complete the `forward` method of `DecoderLayer`.** The first sub-layer is done for you.

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, hidden_dim):
        super().__init__()
        self.self_attn = MultiHeadAttention(embed_dim, num_heads)
        self.cross_attn = MultiHeadAttention(embed_dim, num_heads)
        self.linear1 = nn.Linear(embed_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, embed_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.norm3 = nn.LayerNorm(embed_dim)

    def forward(self, x, enc_output, tgt_mask=None, src_mask=None):
        # x: (batch_size, tgt_len, embed_dim)            the decoder's own representations
        # enc_output: (batch_size, src_len, embed_dim)   the output of the encoder
        # tgt_mask: mask for the self-attention (causal)
        # src_mask: mask for the cross-attention (hides the padding of the source)
        # Returns x and the cross-attention weights (batch_size, num_heads, tgt_len, src_len).

        # 1) Masked self-attention
        attn_output, _ = self.self_attn(x, x, x, mask=tgt_mask)
        x = self.norm1(x + attn_output)

        # 2) Cross-attention: queries from the decoder, keys and values from the encoder
        cross_output, cross_weights = self.cross_attn(x, enc_output, enc_output, mask=src_mask)
        x = self.norm2(x + cross_output)

        # 3) Feed-forward
        ff_output = self.linear2(F.relu(self.linear1(x)))
        x = self.norm3(x + ff_output)

        return x, cross_weights

In [ ]:
# ✅ Check your decoder layer
torch.manual_seed(0)
layer = DecoderLayer(embed_dim=32, num_heads=4, hidden_dim=64)
x = torch.randn(2, 5, 32)           # 5 target positions
enc_output = torch.randn(2, 7, 32)  # 7 source positions

out, cross_weights = layer(x, enc_output)
assert out.shape == (2, 5, 32), f"the output should have shape (2, 5, 32), got {tuple(out.shape)}"
assert cross_weights.shape == (2, 4, 5, 7), "the cross-attention weights should have shape (batch, heads, tgt_len, src_len) = (2, 4, 5, 7)"

out_other, _ = layer(x, torch.randn(2, 7, 32))
assert not torch.allclose(out, out_other), "the output does not depend on enc_output — are the keys and values of the cross-attention taken from the encoder?"

out.sum().backward()
unused = [name for name, p in layer.named_parameters() if p.grad is None]
assert not unused, f"these parameters are never used in forward: {unused}"
print("Looks good ✅")

**3.2 The cross-attention weights have the shape `(batch, heads, tgt_len, src_len)`, while those of the decoder's self-attention are `(batch, heads, tgt_len, tgt_len)`.** Explain the difference. Why does the *self*-attention of the decoder need a causal mask, but the cross-attention does not?

---

*Answer:*

A weight matrix always has **one row per query and one column per key**. In the decoder's self-attention both come from the target, so the matrix is `tgt_len × tgt_len`. In cross-attention the **queries come from the decoder** ("I am about to write the month — where is it?") and the **keys and values come from the encoder output**, so every target position gets a distribution over the `src_len` source positions.

The causal mask exists to keep the decoder from seeing **target tokens it has not generated yet**. During training the whole target is fed in at once, so without the mask position $t$ could simply look at position $t+1$ and copy the answer. The source, on the other hand, is completely known before the first output token is written — at training time and at inference time — so every target position may look at all of it. The only thing to hide there is padding.

---

## 4. The full model

Three things are still missing: the **embeddings**, the **positions** and the **masks**.

### Masks

Both masks are boolean with `True` = *may attend*, and both are built so that they **broadcast** against the score tensor of shape `(batch_size, num_heads, q_len, k_len)`:

- the **source padding mask** has shape `(batch_size, 1, 1, src_len)` — one entry per key, the same for every head and every query. It is used in the encoder's self-attention *and* in the decoder's cross-attention, because in both the keys are source positions.
- the **causal mask** has shape `(1, 1, tgt_len, tgt_len)` — the same for every example and every head.

**4.1 Complete the two mask functions.** You have built both masks in the previous notebook; only the extra dimensions are new.

Hint: indexing with `None` adds a dimension of size 1: if `m` has shape `(B, S)`, then `m[:, None, None, :]` has shape `(B, 1, 1, S)`.

In [ ]:
def build_padding_mask(src):
    # src: (batch_size, src_len) token ids -> (batch_size, 1, 1, src_len), True for real tokens
    return (src != PAD)[:, None, None, :]


def build_causal_mask(tgt_len):
    # -> (1, 1, tgt_len, tgt_len), True where the key is not later than the query
    return torch.tril(torch.ones(tgt_len, tgt_len, dtype=torch.bool, device=device))[None, None]

In [ ]:
# ✅ Check your masks
m = build_padding_mask(torch.tensor([[5, 6, 7, PAD, PAD], [5, 6, 7, 8, 9]]))
assert m.shape == (2, 1, 1, 5) and m.dtype == torch.bool, "the padding mask should be boolean with shape (batch_size, 1, 1, src_len)"
assert m[0, 0, 0].tolist() == [True, True, True, False, False] and m[1].all(), "True for real tokens, False for PAD"

c = build_causal_mask(4)
assert c.shape == (1, 1, 4, 4) and c.dtype == torch.bool, "the causal mask should be boolean with shape (1, 1, tgt_len, tgt_len)"
assert c[0, 0].tolist() == [[True, False, False, False], [True, True, False, False], [True, True, True, False], [True, True, True, True]]
print("Looks good ✅")

### Encoder, decoder and Transformer

The encoder and the decoder turn token ids into vectors, add the positions, and run the result through their stack of layers. For the positions we use the second common option besides the sinusoids of the previous notebook: **learned position embeddings**, an `nn.Embedding` with one vector per position, indexed by `0, 1, 2, ...`.

**4.2 Complete the `forward` methods of `Encoder` and `Decoder`, and the masks in `Transformer.forward`.**

Hints:
- `torch.arange(seq_len, device=...)` gives the position ids. Their embeddings have shape `(seq_len, embed_dim)` and broadcast over the batch when you add them.
- Every decoder layer returns its cross-attention weights. We keep those of the **last** layer for the plots in Part 6.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers, max_len):
        super().__init__()
        self.embed_dim = embed_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(max_len, embed_dim)
        self.layers = nn.ModuleList([EncoderLayer(embed_dim, num_heads, hidden_dim) for _ in range(num_layers)])

    def forward(self, src, src_mask=None):
        # src: (batch_size, src_len) token ids -> (batch_size, src_len, embed_dim)
        x = self.embedding(src) * math.sqrt(self.embed_dim)  # scaled up, as in the paper

        positions = torch.arange(src.size(1), device=src.device)
        x = x + self.pos_embedding(positions)

        for layer in self.layers:
            x = layer(x, mask=src_mask)

        return x


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers, max_len):
        super().__init__()
        self.embed_dim = embed_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(max_len, embed_dim)
        self.layers = nn.ModuleList([DecoderLayer(embed_dim, num_heads, hidden_dim) for _ in range(num_layers)])
        self.out_proj = nn.Linear(embed_dim, vocab_size)

    def forward(self, tgt, enc_output, tgt_mask=None, src_mask=None):
        # tgt: (batch_size, tgt_len) token ids -> logits (batch_size, tgt_len, vocab_size)
        x = self.embedding(tgt) * math.sqrt(self.embed_dim)

        positions = torch.arange(tgt.size(1), device=tgt.device)
        x = x + self.pos_embedding(positions)

        cross_weights = None
        for layer in self.layers:
            x, cross_weights = layer(x, enc_output, tgt_mask=tgt_mask, src_mask=src_mask)

        return self.out_proj(x), cross_weights


class Transformer(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=4, hidden_dim=128, num_layers=2, max_len=64):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_dim, num_heads, hidden_dim, num_layers, max_len)
        self.decoder = Decoder(vocab_size, embed_dim, num_heads, hidden_dim, num_layers, max_len)

    def forward(self, src, tgt):
        # src: (batch_size, src_len), tgt: (batch_size, tgt_len), both token ids
        # Returns the logits (batch_size, tgt_len, vocab_size) and the cross-attention weights of the last layer.

        src_mask = build_padding_mask(src)
        tgt_mask = build_causal_mask(tgt.size(1))

        enc_output = self.encoder(src, src_mask)
        logits, cross_weights = self.decoder(tgt, enc_output, tgt_mask=tgt_mask, src_mask=src_mask)
        return logits, cross_weights

The model runs — but does it run *correctly*? A Transformer with a broken mask produces outputs of the right shape and trains without complaint. The check below therefore tests the **behaviour** that the masks are supposed to guarantee:

- **Causality:** changing target tokens from position 6 onwards must not change the logits at positions 0–5.
- **Padding:** appending extra `<pad>` tokens to the source must not change anything at all.

In [ ]:
# ✅ Check your model
torch.manual_seed(0)
model = Transformer(VOCAB_SIZE).to(device)
model.eval()
src, tgt = make_batch(train_pairs[:4])

with torch.no_grad():
    logits, cross_weights = model(src, tgt)
    assert logits.shape == (4, 12, VOCAB_SIZE), f"the logits should have shape (4, 12, {VOCAB_SIZE}), got {tuple(logits.shape)}"
    assert cross_weights.shape == (4, 4, 12, src.size(1)), "the cross-attention weights should have shape (batch, heads, tgt_len, src_len)"

    # Positions: the same token at two different positions must not get the same representation
    same_tokens = torch.full((1, 6), stoi["1"], device=device)
    enc = model.encoder(same_tokens)
    assert not torch.allclose(enc[0, 0], enc[0, 1], atol=1e-5), "the encoder ignores positions — did you add the position embeddings?"

    # Causality
    tgt_changed = tgt.clone()
    tgt_changed[:, 6:] = stoi["7"]
    logits_changed, _ = model(src, tgt_changed)
    assert torch.allclose(logits[:, :6], logits_changed[:, :6], atol=1e-5), "the decoder can see the future — check the causal mask"
    assert not torch.allclose(logits[:, 6:], logits_changed[:, 6:], atol=1e-5), "the decoder ignores its own input"

    # Padding
    src_padded = torch.cat([src, torch.full((4, 5), PAD, device=device)], dim=1)
    logits_padded, weights_padded = model(src_padded, tgt)
    assert torch.allclose(logits, logits_padded, atol=1e-5), "extra padding changes the output — check the padding mask (encoder AND cross-attention)"
    assert torch.all(weights_padded[..., -5:] == 0), "the decoder attends to <pad> tokens"

n_params = sum(p.numel() for p in model.parameters())
print(f"The model has {n_params:,} parameters")
assert n_params == 183_336, "unexpected number of parameters"
print("Looks good ✅")

## 5. Training and decoding

### Teacher forcing

At inference time the decoder writes one token at a time and feeds each one back in. During training we do not wait for that: we feed in the **correct** target, shifted by one position, and ask the model to predict the next token at **every position at once**:

```
target          <s>  2  0  2  6  -  0  9  -  1  2  </s>

decoder input   <s>  2  0  2  6  -  0  9  -  1  2            = target without its last token
labels           2   0  2  6  -  0  9  -  1  2  </s>         = target without its first token
```

This is called **teacher forcing**. It only works because of the causal mask: position $t$ sees the correct tokens up to $t$ and nothing after it, which is exactly the situation it will be in at inference time.

**5.1 Complete `train_step`.**

Hints:
- `nn.CrossEntropyLoss` expects logits of shape `(N, num_classes)` and labels of shape `(N,)`. Flatten batch and time into one dimension with `.reshape(-1, VOCAB_SIZE)` and `.reshape(-1)`.
- The loss is created with `ignore_index=PAD`, so padded positions would not count. (Our targets happen to need no padding, but this is how you would always write it.)

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD)


def train_step(model, optimizer, src, tgt):
    # One optimisation step on a batch; returns the loss as a Python float.
    model.train()
    decoder_input = tgt[:, :-1]  # everything but the last token
    labels = tgt[:, 1:]          # everything but the first token

    logits, _ = model(src, decoder_input)
    loss = criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    return loss.item()

### Overfit a single batch first

Before you start a long training run, check that the model can **memorise a handful of examples**. It costs a few seconds. If the loss does not go to (almost) zero on eight examples, something is broken — the shift, a mask, a shape — and no amount of data or training time will fix it.

In [ ]:
# ✅ Check your training step: overfit 8 examples
torch.manual_seed(0)
tiny_model = Transformer(VOCAB_SIZE).to(device)
tiny_optimizer = torch.optim.Adam(tiny_model.parameters(), lr=2e-3)
tiny_pairs = train_pairs[:8]
src, tgt = make_batch(tiny_pairs)

for step in range(1, 151):
    loss = train_step(tiny_model, tiny_optimizer, src, tgt)
    if step % 50 == 0:
        print(f"step {step:>3}: loss {loss:.4f}")

assert loss < 0.05, "the model cannot memorise 8 examples — check the shift of decoder input and labels, and your masks"
print("Looks good ✅")

### Greedy decoding

To translate, the model has to generate. **Greedy decoding** is the simplest way: start the target with `<s>`, run the model, take the most likely next token, append it, and repeat.

**5.2 Complete `greedy_decode`.**

Hints:
- The logits have shape `(batch_size, current_length, vocab_size)`. Only the **last** position predicts a token you do not know yet.
- `argmax(dim=-1, keepdim=True)` keeps the result as a column of shape `(batch_size, 1)`, ready for `torch.cat(..., dim=1)`.

In [ ]:
def greedy_decode(model, src, max_len=12):
    # src: (batch_size, src_len). Returns the generated token ids, (batch_size, max_len), starting with BOS.
    model.eval()
    tgt = torch.full((src.size(0), 1), BOS, device=src.device)
    with torch.no_grad():
        for _ in range(max_len - 1):
            logits, _ = model(src, tgt)
            next_token = logits[:, -1].argmax(dim=-1, keepdim=True)  # (batch_size, 1)
            tgt = torch.cat([tgt, next_token], dim=1)
    return tgt


def translate(model, text):
    src, _ = make_batch([(text, "")])
    return decode(greedy_decode(model, src)[0].tolist())


def exact_match(model, pairs):
    # Fraction of pairs for which the whole output is correct.
    src, tgt = make_batch(pairs)
    return (greedy_decode(model, src, max_len=tgt.size(1)) == tgt).all(dim=1).float().mean().item()

In [ ]:
# ✅ Check your decoding on the model that memorised 8 examples
for source, target in tiny_pairs[:3]:
    print(f"{source:<32} -> {translate(tiny_model, source)}")

assert greedy_decode(tiny_model, src).shape == (8, 12), "the output should have shape (batch_size, max_len)"
assert exact_match(tiny_model, tiny_pairs) == 1.0, "the model has memorised these examples, so greedy decoding should reproduce all of them"
print(f"\nExact match on 1,000 unseen dates: {exact_match(tiny_model, test_pairs):.3f}  (it has only memorised, not learned)")
print("Looks good ✅")

### The training run

**5.3 Run the cell below** to train the model on all 20,000 dates for 10 epochs. After every epoch it prints the **exact match** on the test set: the fraction of dates for which *every* character of the output is right. This takes one to two minutes on a CPU.

In [ ]:
BATCH_SIZE = 128
NUM_EPOCHS = 10

torch.manual_seed(0)
model = Transformer(VOCAB_SIZE).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

losses = []
for epoch in range(1, NUM_EPOCHS + 1):
    random.shuffle(train_pairs)
    for i in range(0, len(train_pairs), BATCH_SIZE):
        src, tgt = make_batch(train_pairs[i:i + BATCH_SIZE])
        losses.append(train_step(model, optimizer, src, tgt))
    steps_per_epoch = len(losses) // epoch
    print(f"epoch {epoch:>2}: loss {sum(losses[-steps_per_epoch:]) / steps_per_epoch:.4f}   exact match on the test set {exact_match(model, test_pairs):.3f}")

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.yscale("log")
plt.xlabel("training step")
plt.ylabel("loss")
plt.title("Training loss")
plt.show()

In [ ]:
# ✅ Check the result
accuracy = exact_match(model, test_pairs)
assert accuracy > 0.9, f"the model should translate more than 90% of the test dates correctly, got {accuracy:.1%}"

for source, target in test_pairs[:8]:
    prediction = translate(model, source)
    print(f"{source:<32} -> {prediction}   {'✓' if prediction == target else '✗ expected ' + target}")
print("Looks good ✅")

**5.4 During training the model processes all 11 positions of a target in a single forward pass. `greedy_decode` needs 11 forward passes for the same 11 tokens.** Why can the two not work the same way? And what would happen to the training loss — and to the translations — if you trained *without* the causal mask?

---

*Answer:*

During **training** the correct target is known, so the input of every decoder position is known in advance (teacher forcing). All positions can be computed in parallel, and the causal mask makes sure that position $t$ still only uses tokens $\le t$. At **inference** there is no target: the input at position $t+1$ *is* the model's own prediction at position $t$, which does not exist until position $t$ has been computed. Generation is sequential by nature. (Real implementations at least avoid recomputing everything in every pass by caching the keys and values of earlier positions.)

Without the causal mask, position $t$ can attend to position $t+1$ of the decoder input — which is exactly the label it has to predict. The model learns to **copy from the future**: the training loss drops to zero almost immediately, and it looks like a great success. At inference time there is no future to copy from, and the output is garbage. Nothing crashes and no shape is wrong, which is why the causality test in Part 4 is worth having.

This parallel training is the big practical advantage over last week's RNNs, which need one sequential step per token during training as well.

---

## 6. What did the model learn?

The cross-attention weights tell you, for every character the decoder writes, **which source characters it looked at**. For a translation model this is a soft *alignment* between input and output.

**6.1 Complete the cell below** to get the cross-attention weights for one date and plot all four heads of the last decoder layer.

Hints:
- Run the model on the source and the target **without its last token**, as in training. The rows of the weight matrix then correspond to the tokens the model *predicts*, which are `target_tokens[1:]`.
- `cross_weights` has shape `(batch_size, num_heads, tgt_len, src_len)`, with a batch size of 1 here.

In [ ]:
def plot_cross_attention(weights, source_tokens, predicted_tokens, title=""):
    # weights: (num_heads, tgt_len, src_len) — one heatmap per head
    num_heads = weights.size(0)
    fig, axs = plt.subplots(1, num_heads, figsize=(4.5 * num_heads, 4), sharey=True)
    for head, ax in enumerate(axs):
        ax.imshow(weights[head].detach().cpu(), cmap="Blues", vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(len(source_tokens)), source_tokens, fontsize=8)
        ax.set_yticks(range(len(predicted_tokens)), predicted_tokens)
        ax.set_xlabel("source (attended to)")
        ax.set_title(f"head {head}")
    axs[0].set_ylabel("output (being written)")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def show_attention(model, source, target):
    src, tgt = make_batch([(source, target)])
    model.eval()
    with torch.no_grad():
        _, cross_weights = model(src, tgt[:, :-1])

    source_tokens = [itos[i] for i in src[0].tolist()]
    predicted_tokens = [itos[i] for i in tgt[0, 1:].tolist()]
    plot_cross_attention(cross_weights[0], source_tokens, predicted_tokens, title=f"{source}  ->  {target}")


show_attention(model, "saturday, 12 september 2026", "2026-09-12")

**6.2 Look at a few more dates in other formats.** Change the examples below as you like.

In [ ]:
show_attention(model, "september 12, 2026", "2026-09-12")
show_attention(model, "3.4.1987", "1987-04-03")

**6.3 Describe what you see.** Where does the model look while it writes the year, the month and the day? Does it ever look at the weekday? Do all four heads do the same thing? And how does the picture differ from the neat diagonal you would expect if the output were just a copy of the input?

---

*Answer:*

*(From our run. The details differ from run to run, the overall picture should not.)*

- **Year.** While the model writes `2026`, all heads look at the **far right end** of the source — although the year is the *first* thing it writes — and they move through the four digits roughly one at a time: this part is copying.
- **Month.** For `09` the attention jumps into the month name, and lands sharply on single letters: for `september` mostly on the **`p`**, the third letter. That makes sense. The first letter does not identify a month (`j`: january, june, july; `m`: march, may; `a`: april, august), the first three do — and three letters is all the model gets in the `sep 12 2026` formats, so that is what it learned to read.
- **Day.** For `12` the attention jumps back to the two digits of the day, again one after the other. In `3.4.1987`, the leading zeros of `04` and `03` are not in the source at all: in those rows the heads look at `<s>` and at the separators around the number — that is how the model finds out that the field has only one digit.
- **Weekday.** `saturday,` gets practically no weight anywhere. The model has learned that it carries no information.
- **Heads.** They overlap a lot, with differences in the details (one head concentrates on a single digit where another spreads its weight over the whole year). This is a small model on an easy task, so there is little pressure to specialise.

Instead of a diagonal there are **blocks**: right end → middle → left of the middle. The order in which the source is read follows what the *output* needs next, not the order of the input. An encoder–decoder without attention would have to push all of this through one fixed vector.

---

**6.4 Try to break the model.** It gets practically every test date right — but the test dates come from the same generator as the training data. Translate a few inputs that a person would have no trouble with, but that the model has never seen the like of. Some ideas: a format that is not in `FORMATS` (year first? `12th`?), a year outside 1900–2099, a date that does not exist, a month abbreviated differently.

In [ ]:
for text in ["12 september 2026", "2026 september 12", "12th of september 2026", "12 sept 2026", "12 september 2250", "31 february 2026", "12-09-2026"]:
    print(f"{text:<28} -> {translate(model, text)}")

**6.5 What did you find?** Which changes does the model survive, and which ones break it? What does that tell you about what the model has learned — and about what a test set drawn from the training distribution can tell you?

---

*Answer:*

*(From our run.)*

| input | output | |
|---|---|---|
| `12 sept 2026` | `2026-09-12` | survives: the month is identified by its first three letters anyway |
| `31 february 2026` | `2026-02-31` | copied without complaint: the model has no idea of a calendar |
| `2026 september 12` | `2012-09-20` | the year is expected at the end |
| `12th of september 2026` | `2026-10-12` | a few extra letters, and the month is wrong |
| `12 september 2250` | `2052-09-12` | every year it has ever written started with `19` or `20` |
| `12-09-2026` | `2066-09-12` | an unknown separator, and even the copied year is wrong |

The model has not learned what a *date* is. It has learned a mapping for **seven templates**: where the year sits, which letters identify the month, which characters separate the fields. Inside that distribution it is practically perfect; one step outside, it fails — and it fails **silently**. Every output above is a well-formed date, produced with the same confidence as a correct one.

The 99.7% exact match was measured on dates drawn from the same generator as the training data. Such a test set tells you how well the model has learned *the training distribution*, and nothing about inputs from anywhere else. The large models of the coming weeks are not different in kind; they have simply seen so much that it is much harder to find the outside.

---

## 7. Optional: the same model with `nn.Transformer`

Now that you have built every part yourself, you may use the ready-made one. PyTorch's `nn.Transformer` contains the encoder and decoder stacks, but **not** the embeddings, the positions, the output layer or the masks — you still have to provide those.

**7.1 Complete `SimpleSeq2SeqTransformer`**, using `nn.Transformer` and your sinusoidal `PositionalEncoding` from the previous notebook (copy it into the cell below).

Hints:
- PyTorch's masks use the **opposite convention** for padding: `src_key_padding_mask` is `True` where a token **is** padding and must be ignored.
- `nn.Transformer.generate_square_subsequent_mask(tgt_len)` builds the causal mask (as a float mask with `-inf` above the diagonal).
- Pass the padding mask twice: as `src_key_padding_mask` for the encoder and as `memory_key_padding_mask` for the cross-attention.
- Return `logits, None`, so that the model can be used with `train_step` and `greedy_decode` from above (`nn.Transformer` does not return attention weights).

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class SimpleSeq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, hidden_dim=128, n_layers=2):
        super().__init__()
        self.d_model = d_model

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=n_heads,
            num_encoder_layers=n_layers,
            num_decoder_layers=n_layers,
            dim_feedforward=hidden_dim,
            dropout=0.0,
            batch_first=True,
        )
        self.output = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt):
        src_emb = self.pos_encoder(self.embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.embedding(tgt) * math.sqrt(self.d_model))

        src_is_pad = src == PAD  # True = ignore (PyTorch's convention)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1), device=tgt.device)

        out = self.transformer(
            src_emb, tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_is_pad,
            memory_key_padding_mask=src_is_pad,
        )
        return self.output(out), None

In [ ]:
# ✅ Check your model — the same behavioural tests as in Part 4
import warnings
warnings.filterwarnings("ignore", message=".*nested tensors.*")  # a harmless PyTorch warning about an internal optimisation

torch.manual_seed(0)
simple_model = SimpleSeq2SeqTransformer(VOCAB_SIZE).to(device)
simple_model.eval()
src, tgt = make_batch(test_pairs[:4])

with torch.no_grad():
    logits, _ = simple_model(src, tgt)
    assert logits.shape == (4, 12, VOCAB_SIZE), "wrong shape of the logits"

    tgt_changed = tgt.clone()
    tgt_changed[:, 6:] = stoi["7"]
    assert torch.allclose(logits[:, :6], simple_model(src, tgt_changed)[0][:, :6], atol=1e-5), "the decoder can see the future — check tgt_mask"

    src_padded = torch.cat([src, torch.full((4, 5), PAD, device=device)], dim=1)
    assert torch.allclose(logits, simple_model(src_padded, tgt)[0], atol=1e-5), "extra padding changes the output — check the two padding masks"
print("Looks good ✅")

**7.2 Train it** with the same loop as in 5.3.

In [ ]:
torch.manual_seed(0)
simple_model = SimpleSeq2SeqTransformer(VOCAB_SIZE).to(device)
simple_optimizer = torch.optim.Adam(simple_model.parameters(), lr=2e-3)
print(f"Parameters: {sum(p.numel() for p in simple_model.parameters()):,}   (your own model: {sum(p.numel() for p in model.parameters()):,})")

for epoch in range(1, NUM_EPOCHS + 1):
    random.shuffle(train_pairs)
    for i in range(0, len(train_pairs), BATCH_SIZE):
        src, tgt = make_batch(train_pairs[i:i + BATCH_SIZE])
        loss = train_step(simple_model, simple_optimizer, src, tgt)
    print(f"epoch {epoch:>2}: exact match on the test set {exact_match(simple_model, test_pairs):.3f}")

**7.3 Compare the two models:** the number of parameters, the training time and the exact match. Are you surprised?

---

*Answer:*

*(From our run.)*

- **Parameters:** 172,840 against 183,336 — slightly *fewer*. This model shares one embedding matrix between source and target, and the sinusoidal positions have no parameters; on the other hand `nn.Transformer` adds a final layer norm after the encoder and after the decoder.
- **Time:** about the same per epoch. It is the same computation.
- **Exact match:** clearly *worse* — about 83% after 10 epochs, where your own model reached 99.7%. It does get there (about 95% after 25 epochs), but more slowly and less smoothly.

On paper the two are the same architecture with the same sizes. What differs are "details": fixed sinusoidal instead of learned positions, a shared embedding, PyTorch's Xavier initialisation of `nn.Transformer`, the two extra layer norms. We did not track down which of them is responsible, and it is *not* a reason to think that `nn.Transformer` is worse: given more epochs it gets there as well. It is the lesson of last week's LSTM-vs-GRU notebook again: before you credit (or blame) an architecture, look at initialisation, optimiser settings and training budget.

---

## 8. MCQ

Answer each question by writing the letter of your choice (A–D) after **Answer:**.

---

### 8.1. Encoder–decoder models

Which of the following statements about encoder–decoder (seq2seq) models is **correct**?

A. The encoder always produces a single hidden vector, which is directly mapped to the final output sequence without a decoder<br>
B. In Transformer-based seq2seq models, the decoder attends to both the previously generated tokens and the encoder's hidden states<br>
C. Encoder–decoder models cannot handle variable-length input or output sequences; they require fixed-length sequences on both sides<br>
D. Attention mechanisms are only useful in the encoder and have no role in the decoder<br>

**Answer:** **B**

Every decoder layer has two attention sub-layers: masked self-attention over the tokens generated so far, and cross-attention over the encoder output — you implemented both in 3.1. A is wrong because the output is produced by a decoder, and a Transformer encoder returns one vector *per source token*, not a single one. C is wrong because handling sequences of different lengths is the whole point of seq2seq models (our sources had 8 to 28 characters). D is wrong because the decoder uses attention twice per layer.

---

### 8.2. Scaled dot-product attention

Which of the following statements about the **scaled dot-product attention** used in Transformers is **correct**?

A. The scaling factor $\sqrt{d_k}$ is applied to the keys before the dot product to reduce the computation cost<br>
B. Multi-head attention simply averages the outputs of multiple attention heads to improve stability<br>
C. The softmax function in attention ensures that the attention weights over all keys for a given query sum to 1<br>
D. Self-attention cannot capture long-range dependencies because the receptive field is limited to local context<br>

**Answer:** **C**

The softmax is taken over the key dimension, so every query gets a probability distribution over the keys. A is wrong: the *scores* are divided by $\sqrt{d_k}$, to keep their variance at 1 and the softmax out of saturation (notebook 0, Part 2) — it has nothing to do with cost. B is wrong: the heads are **concatenated** and passed through a linear layer, not averaged. D is wrong: every position attends to every other position directly, however far away.

---

### 8.3. Attentive encoder–decoder models

Which of the following statements about **attentive encoder–decoder models** is correct?

A. Without attention, the decoder always conditions on the entire sequence of encoder hidden states at every step<br>
B. Attention allows the decoder to dynamically focus on different parts of the encoder's hidden states when generating each output token<br>
C. In attentive encoder–decoder models, attention weights are random and fixed; they are not learned during training<br>
D. The introduction of attention prevents the decoder from using its own previously generated tokens<br>

**Answer:** **B**

That is what you saw in Part 6: while writing the year the decoder looks at the year in the source, while writing the month it looks at the month name. A describes the opposite: *without* attention the decoder only gets a single fixed vector (the encoder's last hidden state), which is exactly the bottleneck attention was invented to remove. C is wrong because the weights are computed from learned projections of the current decoder state and the encoder states — they differ for every input and every output position. D is wrong because the decoder still conditions on its previous outputs.

---

### 8.4. Transformer block

Which of the following statements about a **Transformer block** (as used in the original Transformer architecture) is correct?

A. Each Transformer block consists of a single feed-forward network followed by multi-head attention<br>
B. Residual connections and layer normalization are applied around both the multi-head attention sub-layer and the feed-forward sub-layer<br>
C. The position-wise feed-forward network applies the same parameters to every position, but different positions use different networks<br>
D. In the encoder block, masked self-attention is applied to prevent attending to future tokens<br>

**Answer:** **B**

Every sub-layer is wrapped as `LayerNorm(x + SubLayer(x))`, as in your `EncoderLayer` and `DecoderLayer`. A has the order reversed: attention comes first, then the feed-forward network. C contradicts itself: *position-wise* means one and the same network is applied to every position. D is wrong because the causal mask belongs to the **decoder**; the encoder sees the whole source in both directions (it only masks padding).

---

### 8.5. Self-attention

Which of the following statements about **self-attention** in Transformers is correct?

A. Self-attention computes attention weights only between different sequences in a batch, not within a single sequence<br>
B. In self-attention, queries, keys, and values are all derived from the same input representations<br>
C. Self-attention cannot model long-range dependencies because it only attends to local neighbors<br>
D. The complexity of self-attention is linear in sequence length, making it more efficient than RNNs for long inputs<br>

**Answer:** **B**

That is what the *self* stands for: `self_attn(x, x, x)`. In cross-attention, by contrast, the queries come from the decoder and the keys and values from the encoder. A is wrong: attention works within a sequence, and the sequences of a batch never interact. C is wrong: any two positions are connected directly. D is wrong: every position attends to every position, so time and memory grow **quadratically** with the sequence length, $O(n^2)$. An RNN is linear in the length — its problem is that it cannot be parallelised over time.